# Basic Idea of dropout
Here we radomly generate a ind and make it to zero

In [1]:
import random
def dropout(example_output,droopout_rate):
  while True:
    dropped_out = example_output.count(0)
    dr=dropped_out / len(example_output)    
    if dr>=droopout_rate:
      return example_output
    ind=random.randint(0,len(example_output)-1)
    example_output[ind]=0  

In [2]:
example_output = [0.27, -1.03, 0.67, 0.99, 0.05,-0.37, -2.01, 1.13, -0.07, 0.73]
droopout_rate=0.2
dropout(example_output,droopout_rate)

[0, -1.03, 0, 0.99, 0.05, -0.37, -2.01, 1.13, -0.07, 0.73]

# Bernoli probabilty Build-in
## ``np.random.binomial(1, p, size=output.shape)`` generates a 0/1 mask where each element is 1 with probability p, used to randomly drop neurons.

In [3]:
import numpy as np

dropout_rate = 0.3
example_output = np.array([0.27, -1.03, 0.67, 0.99, 0.05,-0.37, -2.01, 1.13, -0.07, 0.73])

example_output = example_output * np.random.binomial(1, 1-dropout_rate,example_output.shape)

print(example_output)

[ 0.   -1.03  0.67  0.99  0.   -0.   -2.01  1.13 -0.07  0.73]


In [4]:
import numpy as np
dropout_rate = 0.2
example_output = np.array([0.27, -1.03, 0.67, 0.99, 0.05,
-0.37, -2.01, 1.13, -0.07, 0.73])
print(f'sum initial {sum(example_output)}')
sums = []
for i in range(10000):
  example_output2=example_output * np.random.binomial(1,1-dropout_rate,example_output.shape)/(1-dropout_rate)
  sums.append(sum(example_output2))
print(f'mean sum: {np.mean(sums)}')

sum initial 0.36000000000000015
mean sum: 0.3734675000000002


# BackWard
<div style="float: left; width: 100%;text-align: center;">
  <img src="./Image/dropout_back.png" alt="Image 2" style="width: 50%;"/>
</div>

In [5]:
class Layer_dropout:
  def __init__(self,rate):
    # Store rate, we invert it as for example for dropout
    # of 0.1 we need success rate of 0.9
    self.rate=1-rate
  def forward(self,input):
    self.inputs=input
    self.binary_mask= np.random.binomial(1,self.rate,input.shape)/(1-self.rate)

    self.output=self.inputs * np.random.binomial(1,self.rate,input.shape)
  def backward(self,dvalue):
    self.dinputs= dvalue *self.binary_mask

In [6]:
from nnfs.datasets import spiral_data
import numpy as np
import nnfs
nnfs.init()
from My_NN.Layer_Dense import  Layer_Dense
from My_NN.activation import  ReLU
from My_NN.Loss import  Softmax_Loss_CategoricalCrossentropy
from My_NN.optimizer import Adam_Optimizer

In [7]:
# Create dataset
X, y = spiral_data(samples=100, classes=3)
dense1 = Layer_Dense(2, 64)
activation1 = ReLU()
dropout1=Layer_dropout(0.1)
dense2 = Layer_Dense(64, 3)
loss_activation = Softmax_Loss_CategoricalCrossentropy()
optimizer = Adam_Optimizer()

In [8]:
for epoch in range(10001):
  
  dense1.forward(X)
  activation1.forward(dense1.output)
  
  dropout1.forward(activation1.output)
  dense2.forward(dropout1.output)

  dense2.forward(activation1.output)
  loss = loss_activation.forward(dense2.output, y)
  predictions = np.argmax(loss_activation.output, axis=1)
  if len(y.shape) == 2:
    y = np.argmax(y, axis=1)
  accuracy = np.mean(predictions==y)
  
  if not epoch % 100:
    print(f'epoch: {epoch}, ' +    f'acc: {accuracy:.3f}, ' +    f'loss: {loss:.3f}')
    
  # Backward pass
  loss_activation.backward(loss_activation.output, y)
  dense2.backward(loss_activation.dinputs)
  
  dropout1.backward(dense2.dinputs)
  activation1.backward(dropout1.dinputs)
  
  dense1.backward(activation1.dinputs)
  
  # Update weights and biases
  optimizer.pre_update_params()
  optimizer.update_params(dense1)
  optimizer.update_params(dense2)
  optimizer.post_update_params()

epoch: 0, acc: 0.360, loss: 1.099


epoch: 100, acc: 0.420, loss: 1.077
epoch: 200, acc: 0.437, loss: 1.072
epoch: 300, acc: 0.460, loss: 1.066
epoch: 400, acc: 0.477, loss: 1.058
epoch: 500, acc: 0.483, loss: 1.046
epoch: 600, acc: 0.480, loss: 1.030
epoch: 700, acc: 0.493, loss: 1.011
epoch: 800, acc: 0.527, loss: 0.988
epoch: 900, acc: 0.543, loss: 0.967
epoch: 1000, acc: 0.593, loss: 0.945
epoch: 1100, acc: 0.607, loss: 0.924
epoch: 1200, acc: 0.613, loss: 0.905
epoch: 1300, acc: 0.620, loss: 0.889
epoch: 1400, acc: 0.633, loss: 0.875
epoch: 1500, acc: 0.637, loss: 0.862
epoch: 1600, acc: 0.650, loss: 0.851
epoch: 1700, acc: 0.653, loss: 0.840
epoch: 1800, acc: 0.667, loss: 0.830
epoch: 1900, acc: 0.680, loss: 0.820
epoch: 2000, acc: 0.683, loss: 0.810
epoch: 2100, acc: 0.700, loss: 0.800
epoch: 2200, acc: 0.700, loss: 0.792
epoch: 2300, acc: 0.703, loss: 0.782
epoch: 2400, acc: 0.710, loss: 0.773
epoch: 2500, acc: 0.713, loss: 0.763
epoch: 2600, acc: 0.727, loss: 0.755
epoch: 2700, acc: 0.730, loss: 0.746
epoch: 280

# Testing 

In [11]:
x_text, y_test = spiral_data(samples=100, classes=3)
dense1.forward(x_text)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss = loss_activation.forward(dense2.output, y_test)
predictions = np.argmax(loss_activation.output, axis=1)
if len(y_test.shape) == 2:
  y_test= np.argmax(y_test, axis=1)
accuracy = np.mean(predictions==y_test)
print("accuracy is: ",accuracy)
print("Loss is: ",loss)

accuracy is:  0.72
Loss is:  0.66326725


# Implementing bernoulli_mask

In [10]:
import random

def bernoulli_mask(shape, p):
    # Recursive function to handle multi-dim shapes
    if len(shape) == 1:
        return [1 if random.random() < p else 0 for _ in range(shape[0])]
    
    return [bernoulli_mask(shape[1:], p) for _ in range(shape[0])]